# M3-B1 — Mission étoile : préparer la 3ᵉ source pour du RAG — SUJET

> **Bonus optionnel, non noté.** Acerox t'a transmis un corpus de **rapports
> techniques** (`rapports_techniques_2024.zip`, 5 fichiers `.md`). C'est du
> **texte non structuré** : impossible à ranger dans une table SQL. Tu vas le
> **préparer** pour une exploitation par similarité sémantique.

## 🚫 Garde-fou
> **Préparation seulement** : tu t'arrêtes à la **récupération** (retrieval).
> La génération augmentée par un LLM, c'est **M7-B2**. Pour la prédiction de
> défauts (tabulaire), un RAG **n'aide pas** : ce corpus sert l'**aide au
> diagnostic humain**. Embeddings **locaux** (aucune clé API).

## Pré-requis
```bash
# Dézippe rapports_techniques_2024.zip dans  ../data/rapports_techniques_2024/
# Décommente le bloc bonus de requirements.txt puis :
pip install chromadb sentence-transformers
```

## 1. Repérer les rapports (fourni)

Note modèle d'embedding (français):
- `all-MiniLM-L6-v2` est surtout optimisé pour l'anglais.
- Pour un corpus de rapports en français, on utilise un modèle multilingue.
- Choix retenu ici: `paraphrase-multilingual-MiniLM-L12-v2` (bon compromis qualité/vitesse, local).

In [6]:
from pathlib import Path

RAPPORTS = Path("../data/rapports_techniques_2024")
EMBED_MODEL = "paraphrase-multilingual-MiniLM-L12-v2"  # Mieux adapté à un corpus en français

fichiers = sorted(RAPPORTS.glob("*.md"))
print(f"{len(fichiers)} rapports trouvés :")
for f in fichiers:
    print(" -", f.name)
print("Modèle d'embedding:", EMBED_MODEL)

5 rapports trouvés :
 - RT-2026-001_Roubaix_ligne1.md
 - RT-2026-002_Lyon_ligne2.md
 - RT-2026-003_Lyon_ligne4.md
 - RT-2026-004_Lyon_ligne4.md
 - RT-2026-005_Lyon_ligne2.md
Modèle d'embedding: paraphrase-multilingual-MiniLM-L12-v2


## 2. DocumentLoader (fait main)

La fonction charge un fichier `.md` et renvoie un dict avec au moins :
`source`, `reference`, `date` (extraits de l'en-tête via regex) et `texte`.
Pas besoin de LangChain.

In [2]:
import re


def _extraire(pattern: str, texte: str, default: str = "inconnu") -> str:
    m = re.search(pattern, texte, flags=re.IGNORECASE)
    return m.group(1).strip() if m else default


def charger_rapport(path: Path) -> dict:
    texte = path.read_text(encoding="utf-8")

    # L'en-tête utile est sur la première ligne commençant par '>'
    entete_match = re.search(r"^>\s*(.+)$", texte, flags=re.MULTILINE)
    entete = entete_match.group(1) if entete_match else ""

    reference = _extraire(r"Référence\s*:\s*([^|\n]+)", entete)
    date = _extraire(r"Date\s*:\s*([0-9]{4}-[0-9]{2}-[0-9]{2})", entete)

    return {
        "source": str(path),
        "reference": reference,
        "date": date,
        "texte": texte,
    }


docs = [charger_rapport(f) for f in fichiers]
print(f"{len(docs)} documents chargés")
for d in docs[:2]:
    print({k: d[k] for k in ["source", "reference", "date"]})

5 documents chargés
{'source': '..\\data\\rapports_techniques_2024\\RT-2026-001_Roubaix_ligne1.md', 'reference': 'RT-2026-001', 'date': '2026-04-01'}
{'source': '..\\data\\rapports_techniques_2024\\RT-2026-002_Lyon_ligne2.md', 'reference': 'RT-2026-002', 'date': '2026-04-08'}


## 3. Segmentation (chunking)

Implémentation de 2 stratégies :
- par titre Markdown (`##`) ;
- par taille fixe (avec recouvrement).

La cellule ci-dessous compare les deux approches et retient une stratégie pour la suite.

In [3]:
def chunk_par_titre(doc: dict) -> list[dict]:
    texte = doc["texte"]
    reference = doc["reference"]

    headers = list(re.finditer(r"(?m)^##\s+(.+)$", texte))
    chunks = []

    if not headers:
        return [{
            "id": f"{reference}_sec_1",
            "texte": texte.strip(),
            "meta": {
                "source": doc["source"],
                "reference": reference,
                "date": doc["date"],
                "section": "document_complet",
                "strategie": "titre",
            },
        }]

    intro = texte[: headers[0].start()].strip()
    if intro:
        chunks.append({
            "id": f"{reference}_sec_intro",
            "texte": intro,
            "meta": {
                "source": doc["source"],
                "reference": reference,
                "date": doc["date"],
                "section": "introduction",
                "strategie": "titre",
            },
        })

    for i, h in enumerate(headers, start=1):
        titre = h.group(1).strip()
        debut = h.end()
        fin = headers[i].start() if i < len(headers) else len(texte)
        corps = texte[debut:fin].strip()
        chunk_texte = f"## {titre}\n{corps}".strip()

        chunks.append({
            "id": f"{reference}_sec_{i}",
            "texte": chunk_texte,
            "meta": {
                "source": doc["source"],
                "reference": reference,
                "date": doc["date"],
                "section": titre,
                "strategie": "titre",
            },
        })

    return chunks


def chunk_taille_fixe(doc: dict, taille: int = 500, overlap: int = 80) -> list[dict]:
    if taille <= 0:
        raise ValueError("taille doit être > 0")
    if overlap < 0 or overlap >= taille:
        raise ValueError("overlap doit être >= 0 et strictement inférieur à taille")

    texte = doc["texte"].strip()
    reference = doc["reference"]
    chunks = []
    start = 0
    i = 1

    while start < len(texte):
        end = min(start + taille, len(texte))
        fenetre = texte[start:end].strip()
        if fenetre:
            chunks.append({
                "id": f"{reference}_fixe_{i}",
                "texte": fenetre,
                "meta": {
                    "source": doc["source"],
                    "reference": reference,
                    "date": doc["date"],
                    "section": f"caracteres_{start}_{end}",
                    "strategie": "fixe",
                },
            })

        if end == len(texte):
            break

        start = end - overlap
        i += 1

    return chunks


def _longueur_moyenne(chunks: list[dict]) -> float:
    if not chunks:
        return 0.0
    return sum(len(c["texte"]) for c in chunks) / len(chunks)


chunks_titre = [c for d in docs for c in chunk_par_titre(d)]
chunks_fixe = [c for d in docs for c in chunk_taille_fixe(d, taille=500, overlap=80)]

print("Comparaison des stratégies de segmentation")
print(f"- Chunks par titre : {len(chunks_titre)} (longueur moyenne: {_longueur_moyenne(chunks_titre):.1f} car.)")
print(f"- Chunks taille fixe : {len(chunks_fixe)} (longueur moyenne: {_longueur_moyenne(chunks_fixe):.1f} car.)")


Comparaison des stratégies de segmentation
- Chunks par titre : 15 (longueur moyenne: 180.5 car.)
- Chunks taille fixe : 10 (longueur moyenne: 313.5 car.)


Stratégie retenue pour la suite: PAR TITRE (15 chunks)
Justification: les sections des rapports portent le sens métier (incident, diagnostic, suite).

## 4. Embeddings + indexation ChromaDB — **TODO**

Encode chaque chunk avec `SentenceTransformer(EMBED_MODEL)` (local), puis range
les chunks dans une collection ChromaDB **persistante**.

> Pense à **vider** la collection si elle existe déjà (idempotence du notebook).

In [8]:
import chromadb
from sentence_transformers import SentenceTransformer

# Fallback si la variable `chunks` n'est pas définie dans le kernel
chunks_a_indexer = globals().get("chunks")
if chunks_a_indexer is None:
    chunks_a_indexer = globals().get("chunks_titre")
if chunks_a_indexer is None:
    raise ValueError("Aucun chunk disponible. Exécute d'abord la cellule de segmentation (partie 3).")

# 1) Embeddings locaux
modele = SentenceTransformer(EMBED_MODEL)
textes = [c["texte"] for c in chunks_a_indexer]
embeddings = modele.encode(textes, show_progress_bar=False).tolist()

# 2) Base vectorielle persistante
client = chromadb.PersistentClient(path="./chroma_acerox")
collection_name = "rapports_acerox"

# Idempotence: on supprime la collection existante puis on la recrée
try:
    client.delete_collection(name=collection_name)
except Exception:
    pass

col = client.get_or_create_collection(name=collection_name)

ids = [c["id"] for c in chunks_a_indexer]
metadatas = [c["meta"] for c in chunks_a_indexer]

col.add(
    ids=ids,
    documents=textes,
    embeddings=embeddings,
    metadatas=metadatas,
)

print(f"Collection '{collection_name}' prête dans ./chroma_acerox")
print(f"Chunks indexés: {col.count()}")
print(f"Modèle d'embedding utilisé: {EMBED_MODEL}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Collection 'rapports_acerox' prête dans ./chroma_acerox
Chunks indexés: 15
Modèle d'embedding utilisé: paraphrase-multilingual-MiniLM-L12-v2


## 5. Interroger par similarité — **TODO**

In [ ]:
def interroger(question: str, k: int = 3) -> None:
    """
    Interroge la collection ChromaDB par similarité sémantique.
    Affiche les k passages les plus proches, leurs métadonnées et distances.
    """
    # 1) Encoder la question
    question_embedding = modele.encode([question], show_progress_bar=False).tolist()
    
    # 2) Requête par similarité (ChromaDB utilise la distance euclidienne par défaut)
    resultats = col.query(
        query_embeddings=question_embedding,
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )
    
    # 3) Affichage formaté
    print(f"\n{'='*80}")
    print(f"Question: {question}")
    print(f"{'='*80}\n")
    
    if not resultats["documents"] or not resultats["documents"][0]:
        print("Aucun résultat trouvé.")
        return
    
    for i, (doc, meta, distance) in enumerate(
        zip(resultats["documents"][0], 
            resultats["metadatas"][0], 
            resultats["distances"][0]), 
        start=1
    ):
        # Score de similarité inversé : plus la distance est petite, plus c'est similaire
        # Utilise une métrique qui gère la distance euclidienne
        score = 1.0 / (1.0 + distance) * 100  # Score entre 0 et 100
        
        print(f"[{i}] Score: {score:.1f} (distance: {distance:.4f})")
        print(f"    Référence: {meta.get('reference', '?')} | Section: {meta.get('section', '?')}")
        print(f"    Date: {meta.get('date', '?')}")
        print(f"    Texte:\n    {doc[:200]}{'...' if len(doc) > 200 else ''}")
        print()


# Exemples de questions métier à tester :
#interroger("Pourquoi des défauts sur la ligne de laminage ?", k=2)
#interroger("Quels capteurs sont peu fiables ?", k=2)
#interroger("Risque RGPD avec les données opérateurs de l'ERP ?", k=2)
interroger("Que faire en cas de vibration ?", k=2)


Question: Dans quel cas a t-on réaliser un colmatage ?

[1] Score: 9.8 (distance: 9.2057)
    Référence: RT-2026-004 | Section: Défaut qualité lot L6925
    Date: 2026-04-18
    Texte:
    ## Défaut qualité lot L6925
Compte rendu d'intervention de maintenance corrective.

Taux de rebut de 5.7 % sur le lot L6925 (cible < 2 %). Corrélation avec la dérive thermique du 2026-04-18. Lot mis e...

[2] Score: 8.5 (distance: 10.7235)
    Référence: RT-2026-001 | Section: Dérive thermique ligne 1
    Date: 2026-04-01
    Texte:
    ## Dérive thermique ligne 1
Synthèse hebdomadaire des alertes supervision.

Température moyenne relevée à 193 °C sur 6 h, au-dessus du seuil de consigne (180 °C). Cause probable : encrassement de l'éc...



## ✅ Vérification (checklist)

- [ ] J'ai chargé les 5 rapports + leurs métadonnées (loader fait main)
- [ ] J'ai comparé 2 stratégies de **segmentation** et justifié mon choix
- [ ] J'ai indexé les chunks dans ChromaDB avec des embeddings **locaux**
- [ ] Une requête en langage naturel récupère les bons passages
- [ ] J'ai vérifié que le **modèle d'embedding est adapté à la langue du corpus** (les rapports sont en français)
- [ ] J'ai écrit 3 lignes (dans ma note) sur **relationnel vs vectoriel**
- [ ] Je sais où s'arrête la **préparation** vs le **RAG complet** (M7-B2)

## ⭐ Pour aller plus loin
- Filtrer par métadonnée (`where={...}`) pour ne chercher que les rapports récents.
- Brancher un LLM local (Ollama) sur les chunks — mais : est-ce justifié ici ?